<a href="https://colab.research.google.com/github/springboardmentor123g/PlantDocBot/blob/intern-AnshikaSahu/Img_Classificaton.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import kagglehub
import os

path = kagglehub.dataset_download("vipoooool/new-plant-diseases-dataset")
dataset_path = os.path.join(
    path, "New Plant Diseases Dataset(Augmented)", "New Plant Diseases Dataset(Augmented)"
)
train_path = os.path.join(dataset_path, "train")

dataset = datasets.ImageFolder(root=train_path, transform=transforms.ToTensor())
loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=2)


mean = 0.0
std = 0.0
num_samples = 0

for images, _ in loader:
    batch_size = images.size(0)

    mean += images.mean([0, 2, 3]) * batch_size
    std += images.std([0, 2, 3]) * batch_size
    num_samples += batch_size

mean /= num_samples
std /= num_samples

print("Dataset mean:", mean.tolist())
print("Dataset std:", std.tolist())


In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import kagglehub
import os
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# Download dataset
path = kagglehub.dataset_download("vipoooool/new-plant-diseases-dataset")
dataset_path = os.path.join(
    path, "New Plant Diseases Dataset(Augmented)", "New Plant Diseases Dataset(Augmented)"
)
train_path = os.path.join(dataset_path, "train")
valid_path = os.path.join(dataset_path, "valid")


# Transforms or DataLoader
mean=[0.4757, 0.5001, 0.4264]
std=[0.1847, 0.1592, 0.2024]

data_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

train_dataset = datasets.ImageFolder(root=train_path, transform=data_transform)
valid_dataset = datasets.ImageFolder(root=valid_path, transform=data_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"Training images: {len(train_dataset)}")
print(f"Validation images: {len(valid_dataset)}")
print(f"Classes: {train_dataset.classes}")


# CNN model
class PlantCNN(nn.Module):
    def __init__(self, num_classes):
        super(PlantCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.5)

        self.fc1 = nn.Linear(128 * 16 * 16, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(train_dataset.classes)
model = PlantCNN(num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training part
epochs = 12
for epoch in range(epochs):
    model.train()
    train_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_loss /= len(train_dataset)
    train_acc = 100. * correct / total

    # Validation
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    val_loss /= len(valid_dataset)
    val_acc = 100. * val_correct / val_total

    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")



torch.save(model, "/content/drive/MyDrive/plant_cnn_.pth")

print("Model saved as plant_cnn.pth")


In [ ]:
model = PlantCNN(num_classes)
model.load_state_dict(torch.load("plant_cnn.pth"))
model.eval()
